# next — 다음 값 하나만 꺼내기

[전체 목차](../../README.md) · [도구 목차](../README.md) · [이 장 목차](README.md)

**이 장의 목표:** iterator에서 값 하나를 꺼내고, 더 꺼낼 값이 없을 때 기본값을 처리한다.
range·enumerate·map·zip보다 나중에 읽어도 됩니다. 먼저 일반 for문 풀이를 이해하고 짧은 표현과 연결하세요.
각 코드 셀은 독립 실행할 수 있습니다.

## 1. 개념과 모범 코드 — 읽은 다음 위치로 이동한다

`next(iterator)`는 **다음 원소 하나**를 반환합니다. 원소가 int면 int, str이면 str, 튜플이면 tuple이 나옵니다.
직접 쓸 때는 `next(iterator, 기본값)` 형태를 함께 챙기세요. 입력을 다 읽었을 때 기본값을 반환합니다.


In [ ]:
numbers = iter([2, 3])
print(next(numbers, -1))  # 예상 출력: 2
print(next(numbers, -1))  # 예상 출력: 3
print(next(numbers, -1))  # 예상 출력: -1


| 호출 | 반환값 | 남은 원소 |
| --- | --- | --- |
| 첫 번째 | 2 | 3 |
| 두 번째 | 3 | 없음 |
| 세 번째 | 기본값 -1 | 없음 |

원본 리스트에서 2를 삭제하는 `pop`과는 다릅니다. reader의 읽기 위치만 바뀝니다.
원본을 다시 사용하고 싶다면 원본과 iterator를 따로 변수에 보관합니다.


In [ ]:
values = [2, 3]
reader = iter(values)
print(next(reader))  # 예상 출력: 2
print(values)  # 예상 출력: [2, 3]
print(list(reader))  # 예상 출력: [3]


## 2. 입력은 iterator여야 한다

문법은 `next(iterator)` 또는 `next(iterator, default)`입니다. 기본값은 두 번째 위치에 전달합니다.
`iter(리스트)`로 iterator를 얻거나, 이미 iterator인 map·zip 등의 결과를 넣습니다.
next 자체는 조건을 검사하거나 값을 변환하지 않습니다.


In [ ]:
converted = map(int, ['12', '34'])
first = next(converted)
print(first)  # 예상 출력: 12
print(type(first).__name__)  # 예상 출력: int

pairs = zip([1, 2], ['a', 'b'])
print(next(pairs))  # 예상 출력: (1, 'a')
print(list(pairs))  # 예상 출력: [(2, 'b')]


| 표현 | 동작 |
| --- | --- |
| `next([2, 3])` | 리스트는 iterator가 아니므로 `TypeError` |
| `next(iter([2, 3]))` | 2 |
| `next(iter([]))` | 끝이므로 `StopIteration` |
| `next(iter([]), -1)` | -1 |

오류를 직접 확인하고 싶다면 아래 셀을 실행하세요. `try/except`는 예외를 잡아 이름만 출력하므로 다음 셀 실행을 막지 않습니다.


In [ ]:
try:
    next(iter([]))
except StopIteration as error:
    print(type(error).__name__)  # 예상 출력: StopIteration

print(next(iter([]), -1))  # 예상 출력: -1


## 3. 모범 코드 — 조건을 만족하는 첫 수 찾기

정수 리스트에서 처음 나오는 짝수를 반환하고, 없으면 None을 반환합니다.
None은 여기서 **찾지 못했다**는 표시이며 숫자 0과 구분됩니다.


In [ ]:
def first_even_model(numbers):
    return next((x for x in numbers if x % 2 == 0), None)

print(first_even_model([1, 3, 8, 4]))  # 예상 출력: 8
print(first_even_model([1, 3, 5]))  # 예상 출력: None
print(first_even_model([]))  # 예상 출력: None
print(first_even_model([0, 2]))  # 예상 출력: 0


이 한 줄은 두 도구가 역할을 나눕니다.

| 부분 | 역할 |
| --- | --- |
| `(x for x in numbers if x % 2 == 0)` | 입력을 순서대로 검사해 짝수만 제공하는 제너레이터(iterator) |
| `next(..., None)` | 그중 첫 값 하나를 요청. 없으면 None |

기본값 인수도 함께 줄 때는 제너레이터 표현식을 **괄호로 감싸서** 첫 번째 인수로 전달합니다.
안쪽의 조건이 검사하고, next는 값 하나만 요청합니다.

## 4. 한 줄을 일반 for문으로 풀기

다음 코드를 먼저 스스로 작성할 수 있으면 충분합니다.


In [ ]:
def first_even_loop_model(numbers):
    for x in numbers:
        if x % 2 == 0:
            return x
    return None

print(first_even_loop_model([1, 3, 8, 4]))  # 예상 출력: 8
print(first_even_loop_model([1, 3, 5]))  # 예상 출력: None


| 읽은 값 | 짝수인가? | 처리 |
| --- | --- | --- |
| 1 | False | 계속 검사 |
| 3 | False | 계속 검사 |
| 8 | True | 8을 반환하고 함수 종료 |
| 4 | 검사하지 않음 | 이미 첫 짝수를 찾음 |

`return None`은 for문 **밖**에 있습니다. 첫 값이 홀수라고 즉시 None을 반환하면 뒤의 8을 못 찾습니다.
함수 바깥에서 풀 경우에는 결과를 저장하고 `break`로 반복을 끝낼 수 있습니다.


In [ ]:
numbers = [1, 3, 8, 4]
answer = None
for x in numbers:
    if x % 2 == 0:
        answer = x
        break
print(answer)  # 예상 출력: 8


## 5. 코테에 연결 — 나머지가 1이 되는 첫 수

정수 n ≥ 3에서 `n % x == 1`인 가장 작은 자연수 x를 찾습니다.
x=1은 나머지가 0이므로 2부터 시작합니다. 작은 수부터 확인하므로 첫 답이 최솟값입니다.


In [ ]:
def remainder_one_model(n):
    for x in range(2, n):
        if n % x == 1:
            return x

print(remainder_one_model(10))  # 예상 출력: 3
print(remainder_one_model(12))  # 예상 출력: 11


In [ ]:
def remainder_one_next_model(n):
    return next(x for x in range(2, n) if n % x == 1)

print(remainder_one_next_model(10))  # 예상 출력: 3
print(remainder_one_next_model(12))  # 예상 출력: 11


n=10이면 x=2에서 나머지 0, x=3에서 나머지 1이므로 3을 반환합니다.
이 코드에 기본값이 없는 이유는 **n ≥ 3이면 x=n-1이 반드시 답 후보**이기 때문입니다.
문제의 보장이 없으면 첫 짝수 예제처럼 기본값을 정해야 합니다.

## 6. 기본값이 해주는 것과 해주지 않는 것

기본값은 iterator가 **정상적으로 끝났을 때만** 사용됩니다. 다른 오류를 감춰 주지는 않습니다.
`next(map(int, ['a']), None)`은 원소는 있지만 정수 변환에 실패하므로 `ValueError`입니다.

기본값은 답과 구분될 값을 고릅니다. 답에 -1이 있을 수 있다면 -1은 적절한 없음 표시가 아닐 수 있습니다.
입력에 None이 없다고 보장된 정수 리스트에서는 None이 편합니다.


In [ ]:
answer = next((x for x in [0, 2] if x % 2 == 0), None)
print(answer)  # 예상 출력: 0
print(answer is None)  # 예상 출력: False


`if not answer`로 실패를 판단하면 정상 답 0까지 실패로 취급합니다. None으로 표시했다면 `answer is None`으로 확인하세요.

## 7. 가져갈 것

1. next는 조건 검색 함수가 아니라 **다음 값 하나를 꺼내는 함수**다.
2. 끝날 수 있다면 기본값을 정한다. 기본값이 없으면 `StopIteration`이다.
3. 첫 값 검색은 일반 for문과 return부터 이해한다. 짧게 쓴다고 계산량이 사라지지는 않는다.

리스트 iterator에서 next 한 번은 O(1)입니다. 그러나 제너레이터가 답을 찾기 위해 k개를 검사하면
`next(제너레이터)` 한 번에도 O(k)가 걸립니다(원소당 조건 검사를 O(1)로 볼 때).
첫 짝수 검색은 원소 수 n에 대해 최악 O(n), 추가 공간 O(1)입니다.
나머지 문제도 입력 수 n에 대해 최악 O(n)입니다(정수 나머지 연산을 O(1)로 보는 코테 모델).
리스트 컴프리헨션으로 답 후보를 전부 만든 뒤 첫 원소를 읽으면, 첫 답 이후의 후보까지 먼저 계산하고 저장하게 됩니다.

[이 장 학습 순서로 돌아가기](README.md) · [제너레이터 표현식](../06-문법/comprehension.md) · [나머지가 1이 되는 수 복습](../../04-복습노트/나머지가-1이-되는-수-찾기.ipynb)

문법 확인: [Python 공식 문서 — next](https://docs.python.org/3.9/library/functions.html#next)
